# Unit 3 — Deep Q-Learning with Atari Space Invaders

Train and evaluate vanilla DQN on `SpaceInvadersNoFrameskip-v4`. The bundled checkpoint was trained from scratch on CPU with a compact CNN and 42×42 observations. The original RL Baselines3 Zoo configuration and commands are also included for the full one-million-step experiment.

To reproduce evaluation in Colab, extract the assignment archive into the working directory so the model folder is next to the notebook. Run the setup and evaluation cells. Training and publication are optional switches.

[Hugging Face Deep RL Course — Unit 3 hands-on](https://huggingface.co/learn/deep-rl-course/unit3/hands-on)

In [ ]:
%pip -q install "gymnasium==0.29.1" "ale-py==0.10.2" "stable-baselines3==2.3.2" "opencv-python-headless==4.11.0.86" "imageio[ffmpeg]" "huggingface_hub<1"

## Compact DQN training

The environment repeats each chosen action across four emulator frames and stacks four grayscale observations. The DQN uses an experience replay buffer, a target network and epsilon-greedy exploration. The checkpoint is selected on a separate validation set. Evaluation uses unclipped rewards and counts a full game as one episode.

In [ ]:
%%writefile train_unit3.py
import json
import os
from pathlib import Path

import ale_py
import gymnasium as gym
import numpy as np
import torch
from torch import nn
from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage

ENV_ID = "SpaceInvadersNoFrameskip-v4"
TRAIN_SEED = 42
MAX_TIMESTEPS = 400_000
STOP_AFTER = int(os.environ.get("UNIT3_STOP_AFTER", MAX_TIMESTEPS))
OUTPUT = Path("unit3-run")
SCREEN_SIZE = 42
DEVICE = os.environ.get("UNIT3_DEVICE", "cpu")
torch.set_num_threads(int(os.environ.get("UNIT3_THREADS", "1")))

class CompactAtariCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=128):
        super().__init__(observation_space, features_dim)
        self.cnn = nn.Sequential(
            nn.Conv2d(observation_space.shape[0], 16, kernel_size=6, stride=3), nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, stride=1), nn.ReLU(), nn.Flatten(),
        )
        with torch.no_grad():
            flat_size = self.cnn(torch.zeros(1, *observation_space.shape)).shape[1]
        self.linear = nn.Sequential(nn.Linear(flat_size, features_dim), nn.ReLU())

    def forward(self, observations):
        return self.linear(self.cnn(observations))

def make_env(training=False, render=False):
    def init_env():
        env = gym.make(ENV_ID, render_mode="rgb_array" if render else None)
        return Monitor(AtariWrapper(env, screen_size=SCREEN_SIZE, terminal_on_life_loss=training, clip_reward=training))
    return VecTransposeImage(VecFrameStack(DummyVecEnv([init_env]), n_stack=4))

def evaluate(model, seeds, render=False, max_steps=None):
    env = make_env(render=render)
    frames, returns, lengths = [], [], []
    try:
        for seed in seeds:
            env.seed(seed)
            obs = env.reset()
            done, total, length = False, 0.0, 0
            if render:
                frames.append(env.render())
            while not done and (max_steps is None or length < max_steps):
                action, _ = model.predict(obs, deterministic=True)
                obs, rewards, dones, infos = env.step(action)
                done = bool(dones[0])
                total += float(rewards[0])
                length += 1
                if render and not done:
                    frames.append(env.render())
            returns.append(total)
            lengths.append(length)
    finally:
        env.close()
    return np.asarray(returns), lengths, frames

def make_model(env):
    return DQN(
        "CnnPolicy", env, learning_rate=1e-4, buffer_size=60_000,
        learning_starts=8_000, batch_size=32, gamma=0.99, train_freq=4,
        gradient_steps=1, target_update_interval=1000, exploration_fraction=0.35,
        exploration_final_eps=0.02, optimize_memory_usage=True,
        replay_buffer_kwargs={"handle_timeout_termination":False},
        policy_kwargs={"features_extractor_class":CompactAtariCNN, "net_arch":[64]},
        seed=TRAIN_SEED, device=DEVICE, verbose=0,
    )

class CheckpointCallback(BaseCallback):
    def __init__(self, directory):
        super().__init__()
        self.directory = directory
        self.best = float("-inf")

    def _on_step(self):
        if self.num_timesteps % 20_000:
            return True
        scores, _, _ = evaluate(self.model, range(1100, 1110))
        mean, std = float(scores.mean()), float(scores.std())
        score = mean - std
        print(f"steps={self.num_timesteps} validation={mean:.2f} +/- {std:.2f}; score={score:.2f}", flush=True)
        if score > self.best:
            self.best = score
            self.model.save(self.directory / "best")
            (self.directory / "validation.json").write_text(json.dumps({"steps":self.num_timesteps,"returns":scores.tolist(),"score":score}))
        return score < 245 and self.num_timesteps < STOP_AFTER

def train():
    OUTPUT.mkdir(parents=True, exist_ok=True)
    env = make_env(training=True)
    try:
        model = make_model(env)
        model.learn(total_timesteps=MAX_TIMESTEPS, callback=CheckpointCallback(OUTPUT))
    finally:
        env.close()

if __name__ == "__main__":
    train()

In [ ]:
%%writefile package_unit3.py
import json
import shutil
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
from stable_baselines3 import DQN
from train_unit3 import ENV_ID, OUTPUT, SCREEN_SIZE, evaluate, make_env

CHECKPOINT = OUTPUT / "best.zip"
MODEL_NAME = f"dqn-{ENV_ID}"
TARGET = Path(MODEL_NAME)

def evaluate_budget(model, budget=5_000):
    env = make_env()
    completed, total = [], 0.0
    try:
        env.seed(4400)
        obs = env.reset()
        for _ in range(budget):
            action, _ = model.predict(obs, deterministic=True)
            obs, rewards, dones, _ = env.step(action)
            total += float(rewards[0])
            if dones[0]:
                completed.append(total)
                total = 0.0
    finally:
        env.close()
    return {"agent_steps": budget, "completed_episode_returns": completed, "partial_episode_return": total}

def main():
    model = DQN.load(CHECKPOINT, device="cpu")
    validation = json.loads((OUTPUT / "validation.json").read_text())
    assert model.num_timesteps == validation["steps"]
    seeds = list(range(2200, 2210))
    rewards, lengths, _ = evaluate(model, seeds)
    additional, _, _ = evaluate(model, range(3300, 3330))
    mean, std = float(rewards.mean()), float(rewards.std())
    score = mean - std
    print(f"Checkpoint {model.num_timesteps:,} steps: {mean:.2f} ± {std:.2f}, score {score:.2f}", flush=True)
    print(f"Additional 30-episode score: {additional.mean()-additional.std():.2f}", flush=True)
    TARGET.mkdir(exist_ok=True)
    shutil.copyfile(CHECKPOINT, TARGET / f"{MODEL_NAME}.zip")
    replay, replay_lengths, frames = evaluate(model, seeds[:1], render=True)
    np.testing.assert_array_equal(replay, rewards[:1])
    assert replay_lengths == lengths[:1]
    with imageio.get_writer(TARGET / "replay.mp4", fps=15, codec="libx264", macro_block_size=2,
                            ffmpeg_log_level="error") as writer:
        for frame in frames:
            writer.append_data(frame)
    record = {
        "env_id": ENV_ID, "train_steps": model.num_timesteps, "mean_reward": mean,
        "std_reward": std, "score": score, "n_eval_episodes": len(seeds),
        "evaluation_seeds": seeds, "episode_rewards": rewards.tolist(), "episode_lengths": lengths,
        "deterministic_policy": True,
        "additional_30_episode_evaluation": {"mean": float(additional.mean()), "std": float(additional.std()),
                                            "seeds": list(range(3300, 3330)), "rewards": additional.tolist()},
        "five_thousand_step_evaluation": evaluate_budget(model),
        "validation": validation,
    }
    (TARGET / "results.json").write_text(json.dumps(record, indent=2) + "\n")
    (TARGET / "config.json").write_text(json.dumps({
        "env_id": ENV_ID, "algorithm": "DQN", "screen_size": SCREEN_SIZE, "frame_stack": 4,
        "gymnasium": "0.29.1", "ale_py": "0.10.2", "stable_baselines3": "2.3.2",
        "terminal_on_life_loss_for_evaluation": False, "clip_reward_for_evaluation": False,
    }, indent=2) + "\n")
    (TARGET / "README.md").write_text(f"""---
library_name: stable-baselines3
tags:
- {ENV_ID}
- deep-reinforcement-learning
- reinforcement-learning
- stable-baselines3
- dqn
model-index:
- name: {MODEL_NAME}
  results:
  - task:
      type: reinforcement-learning
      name: reinforcement-learning
    dataset:
      type: {ENV_ID}
      name: {ENV_ID}
    metrics:
    - type: mean_reward
      name: mean_reward
      value: "{mean:.2f} +/- {std:.2f}"
---

# DQN for Space Invaders

Vanilla Stable-Baselines3 DQN trained from random initialization on `{ENV_ID}` for the [Hugging Face Deep RL Course, Unit 3](https://huggingface.co/learn/deep-rl-course/unit3/hands-on).

The selected checkpoint was saved at **{model.num_timesteps:,} agent timesteps**. The training run used a compact CNN, 42×42 grayscale observations, four stacked frames, and a fixed training seed. Evaluation uses full games and unclipped rewards. Checkpoint selection and evaluation use separate seeds.

| Evaluation | Mean reward | Population standard deviation | Score |
| --- | ---: | ---: | ---: |
| 10 episodes | {mean:.2f} | {std:.2f} | **{score:.2f}** |
| Additional 30 episodes | {additional.mean():.2f} | {additional.std():.2f} | {additional.mean()-additional.std():.2f} |

The score is mean reward minus standard deviation. `results.json` also includes a separate 5,000-agent-step evaluation. `replay.mp4` shows the first of the 10 evaluation games.

Use the evaluation environment settings in `config.json` or run the included training code. Only load model archives from sources you trust.
""", encoding="utf-8")
    shutil.copyfile("train_unit3.py", TARGET / "train_unit3.py")
    print(f"Replay: {replay[0]:g}; 5,000-step completed games: {len(record['five_thousand_step_evaluation']['completed_episode_returns'])}", flush=True)

if __name__ == "__main__":
    main()

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

import torch

TRAIN_FROM_SCRATCH = False
if TRAIN_FROM_SCRATCH:
    subprocess.run(
        [sys.executable, "train_unit3.py"],
        env={**os.environ, "UNIT3_DEVICE": "cpu", "UNIT3_STOP_AFTER": "20000"},
        check=True,
    )
    subprocess.run([sys.executable, "package_unit3.py"], check=True)

## Evaluate the saved agent

The primary evaluation runs ten deterministic full games on held-out seeds. The course score is the mean reward minus the population standard deviation, with a threshold of 200. A separate evaluation runs for exactly 5,000 agent decisions, as in the course walkthrough.

In [1]:
from IPython.display import Video, display
from stable_baselines3 import DQN
from train_unit3 import ENV_ID, evaluate
from package_unit3 import evaluate_budget

repo_folder = Path(f"dqn-{ENV_ID}")
checkpoint = repo_folder / f"dqn-{ENV_ID}.zip"
if not checkpoint.is_file():
    raise FileNotFoundError("Extract the assignment archive into the current directory first.")
model = DQN.load(checkpoint, device="cpu")
rewards, lengths, _ = evaluate(model, range(2200, 2210))
mean, std = rewards.mean(), rewards.std()
print(f"Checkpoint: {model.num_timesteps:,} agent steps")
print(f"Rewards: {rewards.tolist()}")
print(f"Mean reward: {mean:.2f}; standard deviation: {std:.2f}; score: {mean-std:.2f}")
print(evaluate_budget(model))
display(Video(str(repo_folder / "replay.mp4"), embed=True, width=400))

Checkpoint: 20,000 agent steps
Rewards: [210.0, 210.0, 210.0, 210.0, 210.0, 210.0, 210.0, 210.0, 210.0, 210.0]
Mean reward: 210.00; standard deviation: 0.00; score: 210.00
{'agent_steps': 5000, 'completed_episode_returns': [210.0, 210.0, 210.0, 210.0, 210.0, 210.0, 210.0, 210.0], 'partial_episode_return': 0.0}


## Original course workflow with RL Baselines3 Zoo

The original configuration uses the standard CNN and a training budget of one million agent decisions. The commands use the course environment ID. With these versions of Gymnasium and ALE, the CLI needs `--gym-packages ale_py` to register Atari. The command sequence was tested locally with a short smoke run.

In [ ]:
%%writefile dqn.yml
SpaceInvadersNoFrameskip-v4:
  env_wrapper:
    - stable_baselines3.common.atari_wrappers.AtariWrapper
  frame_stack: 4
  policy: CnnPolicy
  n_timesteps: 1000000
  buffer_size: 100000
  learning_rate: 0.0001
  batch_size: 32
  learning_starts: 100000
  target_update_interval: 1000
  train_freq: 4
  gradient_steps: 1
  exploration_fraction: 0.1
  exploration_final_eps: 0.01
  optimize_memory_usage: False

In [ ]:
RUN_ORIGINAL_ZOO = False

if RUN_ORIGINAL_ZOO:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "rl-zoo3==2.3.0",
         "sb3-contrib==2.3.0", "stable-baselines3==2.3.2"],
        check=True,
    )
    device = "cuda" if torch.cuda.is_available() else "cpu"
    args = ["--gym-packages", "ale_py", "--algo", "dqn", "--env", ENV_ID, "--device", device, "--seed", "42"]
    subprocess.run(
        [sys.executable, "-m", "rl_zoo3.train", *args, "-f", "logs/", "-c", "dqn.yml"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "rl_zoo3.enjoy", *args, "--no-render",
         "--n-timesteps", "5000", "--folder", "logs/"],
        check=True,
    )

## Publish the trained agent

To publish the bundled checkpoint, set the switch below to True and run the cell. Use the official Hugging Face sign-in prompt for a write token. The upload is public and includes the model, course metadata, evaluation results, replay and training code. Then check the [course progress tracker](https://huggingface.co/spaces/ThomasSimonini/Check-my-progress-Deep-RL-Course).

For an original Zoo run, the course's separate `rl_zoo3.push_to_hub` command packages the model in `logs/`. Import `ale_py` in the CLI process before invoking that module.

In [ ]:
PUBLISH_TO_HUB = False

if PUBLISH_TO_HUB:
    from huggingface_hub import HfApi, notebook_login

    notebook_login(add_to_git_credential=False)
    api = HfApi()
    repo_id = f"{api.whoami()['name']}/{repo_folder.name}"
    api.create_repo(repo_id, repo_type="model", private=False, exist_ok=True)
    api.upload_folder(repo_id=repo_id, folder_path=repo_folder,
                      commit_message="Publish Unit 3 Space Invaders DQN and evaluation")
    print(f"https://huggingface.co/{repo_id}")

### Optional course example — load a published Beam Rider model

After installing RL Zoo, these commands download the course example and evaluate it for 5,000 agent steps. Use checkpoints from sources you trust.

```bash
python -m rl_zoo3.load_from_hub --algo dqn --env BeamRiderNoFrameskip-v4 -orga sb3 -f rl_trained/
python -m rl_zoo3.enjoy --gym-packages ale_py --algo dqn --env BeamRiderNoFrameskip-v4 -n 5000 -f rl_trained/ --no-render
```